#Notebook 2: Delta Memory Manager
**agentic-civic-resolution-app**

**Goal:** `CivicOpsMemory` class — all read/write operations against Delta memory tables.
Same API as the Lakebase version — swap back later by changing only this file.

 | Method | What it does |
 |--------|-------------|
 | `save_ticket()` | Persist ticket to complaint_history |
 | `update_status()` | Update status + write to sla_status_history |
 | `check_recurring()` | Detect if complaint is recurring at same location |
 | `save_escalation()` | Log escalation to escalation_context |
 | `get_location_context()` | Past complaints at a location |
 | `get_ticket()` | Full ticket by ticket_id |
 | `get_chronic_locations()` | All chronic problem spots |
 | `get_sla_breached()` | All open tickets past SLA deadline |

## 0. Imports

In [0]:
import json, re
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F
from pyspark.sql import Row
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType

CATALOG = "civicops"
SCHEMA  = "memory"

CH  = f"{CATALOG}.{SCHEMA}.complaint_history"
RC  = f"{CATALOG}.{SCHEMA}.recurring_complaints"
EC  = f"{CATALOG}.{SCHEMA}.escalation_context"
SLA = f"{CATALOG}.{SCHEMA}.sla_status_history"

print("✓ Memory table references ready.")

## 1. Memory Manager Class

In [0]:

class CivicOpsMemory:
    """
    Delta-backed persistent memory for CivicOps AI.
    Drop-in replacement for the Lakebase version — same public API.
    """

    # ── 1. Save Ticket ────────────────────────────────────────────────────────
    def save_ticket(self, ticket: dict) -> str:
        """
        Upsert a CivicOpsTicket into complaint_history.
        Automatically triggers recurring + escalation tracking.
        """
        location_hint = self._extract_location(ticket.get("raw_complaint", ""))

        row = {
            "ticket_id":                   ticket["ticket_id"],
            "raw_complaint":               ticket.get("raw_complaint"),
            "category":                    ticket.get("category"),
            "subcategory":                 ticket.get("subcategory"),
            "classification_confidence":   float(ticket.get("classification_confidence", 0)),
            "severity_score":              int(ticket.get("severity_score", 1)),
            "priority_label":              ticket.get("priority_label"),
            "sla_hours":                   int(ticket.get("sla_hours", 72)),
            "sla_deadline":                ticket.get("sla_deadline"),
            "health_risk":                 bool(ticket.get("health_risk", False)),
            "infrastructure_risk":         bool(ticket.get("infrastructure_risk", False)),
            "affected_estimate":           ticket.get("affected_estimate"),
            "dept_code":                   ticket.get("dept_code"),
            "dept_name":                   ticket.get("dept_name"),
            "dept_contact":                ticket.get("dept_contact"),
            "officer_tier":                ticket.get("officer_tier"),
            "escalate":                    bool(ticket.get("escalate", False)),
            "escalation_dept":             ticket.get("escalation_dept"),
            "action_plan":                 json.dumps(ticket.get("action_plan", [])),
            "field_visit_needed":          bool(ticket.get("field_visit_needed", False)),
            "notify_citizen":              bool(ticket.get("notify_citizen", False)),
            "status":                      "OPEN",
            "location_hint":               location_hint,
            "processing_ms":               int(ticket.get("processing_ms", 0)),
            "created_at":                  datetime.now(timezone.utc),
            "updated_at":                  datetime.now(timezone.utc),
            "resolved_at":                 None,
        }

        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType

        COMPLAINT_SCHEMA = StructType([
            StructField("ticket_id",                  StringType(),   True),
            StructField("raw_complaint",              StringType(),   True),
            StructField("category",                   StringType(),   True),
            StructField("subcategory",                StringType(),   True),
            StructField("classification_confidence",  DoubleType(),   True),
            StructField("severity_score",             IntegerType(),  True),
            StructField("priority_label",             StringType(),   True),
            StructField("sla_hours",                  IntegerType(),  True),
            StructField("sla_deadline",               TimestampType(),True),
            StructField("health_risk",                BooleanType(),  True),
            StructField("infrastructure_risk",        BooleanType(),  True),
            StructField("affected_estimate",          StringType(),   True),
            StructField("dept_code",                  StringType(),   True),
            StructField("dept_name",                  StringType(),   True),
            StructField("dept_contact",               StringType(),   True),
            StructField("officer_tier",               StringType(),   True),
            StructField("escalate",                   BooleanType(),  True),
            StructField("escalation_dept",            StringType(),   True),
            StructField("action_plan",                StringType(),   True),
            StructField("field_visit_needed",         BooleanType(),  True),
            StructField("notify_citizen",             BooleanType(),  True),
            StructField("status",                     StringType(),   True),
            StructField("location_hint",              StringType(),   True),
            StructField("processing_ms",              IntegerType(),  True),
            StructField("created_at",                 TimestampType(),True),
            StructField("updated_at",                 TimestampType(),True),
            StructField("resolved_at",                TimestampType(),True),
        ])

        new_df = spark.createDataFrame([row], schema=COMPLAINT_SCHEMA)

        # Upsert — update if ticket_id exists, insert if new
        if spark.catalog.tableExists(CH):
            dt = DeltaTable.forName(spark, CH)
            dt.alias("existing").merge(
                new_df.alias("new"),
                "existing.ticket_id = new.ticket_id"
            ).whenMatchedUpdate(set={
                "updated_at": F.current_timestamp(),
                "status":     "existing.status",   # don't overwrite status on re-run
            }).whenNotMatchedInsertAll().execute()
        else:
            new_df.write.format("delta").mode("append").saveAsTable(CH)

        # Side effects
        self._update_recurring(ticket, location_hint)
        if ticket.get("escalate"):
            self.save_escalation({
                "ticket_id":              ticket["ticket_id"],
                "escalated_from_dept":    ticket.get("dept_code"),
                "escalated_to_dept":      ticket.get("escalation_dept"),
                "escalation_reason":      f"Severity {ticket.get('severity_score')} / health_risk={ticket.get('health_risk')}",
                "severity_at_escalation": ticket.get("severity_score"),
                "health_risk":            ticket.get("health_risk"),
                "infrastructure_risk":    ticket.get("infrastructure_risk"),
                "notes":                  None,
            })

        return ticket["ticket_id"]

    # ── 2. Update Status ──────────────────────────────────────────────────────
    def update_status(
        self,
        ticket_id:  str,
        new_status: str,
        changed_by: str = "system",
        notes:      str = None,
    ) -> dict:
        """Update ticket status and write SLA audit entry."""
        # Get current ticket
        row = spark.table(CH).filter(F.col("ticket_id") == ticket_id).first()
        if not row:
            raise ValueError(f"Ticket {ticket_id} not found in memory.")

        now          = datetime.now(timezone.utc)
        sla_deadline = row["sla_deadline"]
        sla_breached = sla_deadline is not None and now > sla_deadline.replace(tzinfo=timezone.utc)
        hours_remaining = None
        if sla_deadline:
            delta = sla_deadline.replace(tzinfo=timezone.utc) - now
            hours_remaining = round(delta.total_seconds() / 3600, 2)

        # Update complaint_history
        dt = DeltaTable.forName(spark, CH)
        update_map = {
            "status":     f"'{new_status}'",
            "updated_at": "current_timestamp()",
        }
        if new_status == "RESOLVED":
            update_map["resolved_at"] = "current_timestamp()"
        dt.update(
            condition=F.col("ticket_id") == ticket_id,
            set={k: F.expr(v) for k, v in update_map.items()}
        )

        # Append to sla_status_history
        sla_row = Row(
            ticket_id=ticket_id,
            previous_status=row["status"],
            new_status=new_status,
            changed_at=now,
            changed_by=changed_by,
            sla_deadline=sla_deadline,
            sla_breached=sla_breached,
            hours_remaining_at_change=hours_remaining,
            notes=notes,
        )
        spark.createDataFrame([sla_row]).write.format("delta").mode("append").saveAsTable(SLA)

        # If resolved, update recurring unresolved count
        if new_status == "RESOLVED":
            self._mark_recurring_resolved(ticket_id, row["category"], row["location_hint"])

        return {"ticket_id": ticket_id, "new_status": new_status, "sla_breached": sla_breached}

    # ── 3. Check Recurring ────────────────────────────────────────────────────
    def check_recurring(self, location_hint: str, complaint_type: str) -> dict:
        """Returns recurring info for a location + type. Call before processing."""
        location_key = self._normalize_location(location_hint)
        row = (
            spark.table(RC)
            .filter(
                (F.col("location_key") == location_key) &
                (F.col("complaint_type") == complaint_type)
            )
            .first()
        )
        if not row:
            return {"is_recurring": False, "occurrence_count": 0, "is_chronic": False}

        return {
            "is_recurring":      True,
            "occurrence_count":  row["occurrence_count"],
            "avg_severity":      row["avg_severity"],
            "max_severity":      row["max_severity"],
            "is_chronic":        row["is_chronic"],
            "first_reported_at": row["first_reported_at"],
            "last_reported_at":  row["last_reported_at"],
            "resolved_count":    row["resolved_count"],
            "unresolved_count":  row["unresolved_count"],
        }

    # ── 4. Save Escalation ────────────────────────────────────────────────────
    def save_escalation(self, escalation: dict):
        """Log an escalation event to escalation_context."""
        row = Row(
            ticket_id=escalation.get("ticket_id"),
            escalated_at=datetime.now(timezone.utc),
            escalated_from_dept=escalation.get("escalated_from_dept"),
            escalated_to_dept=escalation.get("escalated_to_dept"),
            escalation_reason=escalation.get("escalation_reason"),
            severity_at_escalation=escalation.get("severity_at_escalation"),
            health_risk=escalation.get("health_risk"),
            infrastructure_risk=escalation.get("infrastructure_risk"),
            resolved_after_escalation=False,
            resolution_time_hrs=None,
            notes=escalation.get("notes"),
        )
        
        ESCALATION_SCHEMA = StructType([
            StructField("ticket_id",                  StringType(),   True),
            StructField("escalated_at",               TimestampType(),True),
            StructField("escalated_from_dept",        StringType(),   True),
            StructField("escalated_to_dept",          StringType(),   True),
            StructField("escalation_reason",          StringType(),   True),
            StructField("severity_at_escalation",     IntegerType(),  True),
            StructField("health_risk",                BooleanType(),  True),
            StructField("infrastructure_risk",        BooleanType(),  True),
            StructField("resolved_after_escalation",  BooleanType(),  True),
            StructField("resolution_time_hrs",        DoubleType(),   True),
            StructField("notes",                      StringType(),   True),
        ])

        spark.createDataFrame([row], schema=ESCALATION_SCHEMA).write.format("delta").mode("append").saveAsTable(EC)

    # ── 5. Get Location Context ───────────────────────────────────────────────
    def get_location_context(self, location_hint: str, limit: int = 5) -> list[dict]:
        """Past complaints at a location — gives agents memory of prior incidents."""
        pattern = f"%{self._normalize_location(location_hint)}%"
        rows = (
            spark.table(CH)
            .filter(F.lower(F.col("location_hint")).like(pattern))
            .orderBy(F.col("created_at").desc())
            .limit(limit)
            .select("ticket_id", "category", "subcategory", "severity_score",
                    "status", "created_at", "resolved_at", "escalate", "dept_code")
            .collect()
        )
        return [row.asDict() for row in rows]

    # ── 6. Get Ticket ─────────────────────────────────────────────────────────
    def get_ticket(self, ticket_id: str) -> dict | None:
        row = spark.table(CH).filter(F.col("ticket_id") == ticket_id).first()
        return row.asDict() if row else None

    # ── 7. Get Chronic Locations ──────────────────────────────────────────────
    def get_chronic_locations(self, min_occurrences: int = 3) -> list[dict]:
        rows = (
            spark.table(RC)
            .filter(F.col("is_chronic") | (F.col("occurrence_count") >= min_occurrences))
            .orderBy(F.col("occurrence_count").desc(), F.col("max_severity").desc())
            .collect()
        )
        return [row.asDict() for row in rows]

    # ── 8. Get SLA Breached ───────────────────────────────────────────────────
    def get_sla_breached(self) -> list[dict]:
        rows = (
            spark.table(CH)
            .filter(
                (~F.col("status").isin("RESOLVED", "CLOSED")) &
                (F.col("sla_deadline") < F.current_timestamp())
            )
            .withColumn(
                "hrs_overdue",
                F.round(
                    (F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("sla_deadline")) / 3600,
                    1
                )
            )
            .orderBy(F.col("hrs_overdue").desc())
            .select("ticket_id", "category", "severity_score", "dept_code",
                    "sla_deadline", "created_at", "hrs_overdue")
            .collect()
        )
        return [row.asDict() for row in rows]

    # ── Private Helpers ───────────────────────────────────────────────────────
    def _extract_location(self, text: str) -> str:
        for p in [r'near\s+(.+?)(?:\.|,|$)', r'at\s+(.+?)(?:\.|,|$)', r'on\s+(.+?)(?:\.|,|$)']:
            m = re.search(p, text, re.IGNORECASE)
            if m:
                return m.group(1).strip()[:128]
        return text[:128]

    def _normalize_location(self, location: str) -> str:
        return re.sub(r'\s+', ' ', location.lower().strip())[:128]

    def _update_recurring(self, ticket: dict, location_hint: str):
        """Upsert recurring_complaints when a new ticket is saved."""
        location_key   = self._normalize_location(location_hint)
        complaint_type = ticket.get("category", "Unknown")
        ticket_id      = ticket["ticket_id"]
        severity       = int(ticket.get("severity_score", 1))
        now            = datetime.now(timezone.utc)

        existing = (
            spark.table(RC)
            .filter(
                (F.col("location_key") == location_key) &
                (F.col("complaint_type") == complaint_type)
            )
            .first()
        )

        if not existing:
            new_row = Row(
                location_key=location_key,
                complaint_type=complaint_type,
                first_reported_at=now,
                last_reported_at=now,
                occurrence_count=1,
                avg_severity=float(severity),
                max_severity=severity,
                resolved_count=0,
                unresolved_count=1,
                ticket_ids=json.dumps([ticket_id]),
                is_chronic=False,
                chronic_flagged_at=None,
            )
            
            from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType

            RECURRING_SCHEMA = StructType([
                StructField("location_key",       StringType(),   True),
                StructField("complaint_type",     StringType(),   True),
                StructField("first_reported_at",  TimestampType(),True),
                StructField("last_reported_at",   TimestampType(),True),
                StructField("occurrence_count",   IntegerType(),  True),
                StructField("avg_severity",       DoubleType(),   True),
                StructField("max_severity",       IntegerType(),  True),
                StructField("resolved_count",     IntegerType(),  True),
                StructField("unresolved_count",   IntegerType(),  True),
                StructField("ticket_ids",         StringType(),   True),
                StructField("is_chronic",         BooleanType(),  True),
                StructField("chronic_flagged_at", TimestampType(),True),
            ])

            spark.createDataFrame([new_row],schema=RECURRING_SCHEMA).write.format("delta").mode("append").saveAsTable(RC)

        else:
            new_count   = existing["occurrence_count"] + 1
            new_avg     = round(
                (existing["avg_severity"] * existing["occurrence_count"] + severity) / new_count, 2
            )
            new_max     = max(existing["max_severity"], severity)
            is_chronic  = new_count >= 3
            ticket_ids  = json.loads(existing["ticket_ids"] or "[]") + [ticket_id]
            chronic_at  = existing["chronic_flagged_at"] or (now if new_count == 3 else None)

            dt = DeltaTable.forName(spark, RC)
            dt.update(
                condition=(F.col("location_key") == location_key) &
                          (F.col("complaint_type") == complaint_type),
                set={
                    "occurrence_count":  F.lit(new_count),
                    "last_reported_at":  F.lit(now),
                    "avg_severity":      F.lit(new_avg),
                    "max_severity":      F.lit(new_max),
                    "unresolved_count":  F.col("unresolved_count") + 1,
                    "ticket_ids":        F.lit(json.dumps(ticket_ids)),
                    "is_chronic":        F.lit(is_chronic),
                    "chronic_flagged_at": F.lit(chronic_at),
                }
            )

    def _mark_recurring_resolved(self, ticket_id: str, complaint_type: str, location_hint: str):
        if not location_hint:
            return
        location_key = self._normalize_location(location_hint)
        dt = DeltaTable.forName(spark, RC)
        dt.update(
            condition=(F.col("location_key") == location_key) &
                      (F.col("complaint_type") == complaint_type),
            set={
                "resolved_count":   F.col("resolved_count") + 1,
                "unresolved_count": F.greatest(F.col("unresolved_count") - 1, F.lit(0)),
            }
        )


In [0]:
# Singleton
memory = CivicOpsMemory()
print("✓ CivicOpsMemory (Delta) ready.")

## 2. Smoke Test

In [0]:
from datetime import datetime, timezone, timedelta

_t = {
    "ticket_id":                  "TKT-TEST0001",
    "raw_complaint":              "Garbage overflow near Whitefield junction",
    "category":                   "Sanitation",
    "subcategory":                "Garbage Overflow",
    "classification_confidence":  0.95,
    "severity_score":             3,
    "priority_label":             "Medium",
    "sla_hours":                  72,
    "sla_deadline":               datetime.now(timezone.utc) + timedelta(hours=72),
    "health_risk":                False,
    "infrastructure_risk":        False,
    "affected_estimate":          "10-50 people",
    "dept_code":                  "SWMD",
    "dept_name":                  "Solid Waste Management Dept",
    "dept_contact":               "swmd@civicops.city",
    "officer_tier":               "Field Supervisor Level",
    "escalate":                   False,
    "escalation_dept":            None,
    "action_plan":                ["Dispatch crew", "Clear overflow", "Log completion"],
    "field_visit_needed":         True,
    "notify_citizen":             True,
    "processing_ms":              1200,
}


In [0]:
# Save
tid = memory.save_ticket(_t)
print(f"✓ Saved: {tid}")

# Update status
r = memory.update_status(tid, "IN_PROGRESS", changed_by="field_officer_01", notes="Crew dispatched")
print(f"✓ Status: {r}")

In [0]:
# Second complaint at same location → triggers recurring
_t2 = {**_t, "ticket_id": "TKT-TEST0002"}
memory.save_ticket(_t2)
rec = memory.check_recurring("Whitefield junction", "Sanitation")
print(f"✓ Recurring: occurrences={rec['occurrence_count']}, chronic={rec['is_chronic']}")

In [0]:
# Third → should flip is_chronic = True
_t3 = {**_t, "ticket_id": "TKT-TEST0003"}
memory.save_ticket(_t3)
rec3 = memory.check_recurring("Whitefield junction", "Sanitation")
print(f"✓ After 3rd: chronic={rec3['is_chronic']} (should be True)")

In [0]:
# Location context
ctx = memory.get_location_context("Whitefield")
print(f"✓ Location context: {len(ctx)} records")

# SLA breached
breached = memory.get_sla_breached()
print(f"✓ SLA breached: {len(breached)} tickets")

In [0]:
# Chronic locations
chronic = memory.get_chronic_locations()
print(f"✓ Chronic locations: {len(chronic)}")
for c in chronic:
    print(f"   📍 {c['location_key']} | {c['complaint_type']} | occurrences={c['occurrence_count']}")
